<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/Tesis_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import prince

In [1]:
!pip install prince

In [11]:
# ==========================================
# ⚙️ PASO 1: FIJANDO CEROS A LA IZQUIERDA (ZFILL) Y SUMMARY METRICS
# ==========================================
print("1. Cargando bases de datos en modo Lazy...")
df_bcra = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/deudores_enero_2026_clasificados.parquet")
padron = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/padron_fisicas_enero.parquet")
entidades_maestro = pl.scan_csv("/content/drive/MyDrive/Tesis2026/EntidadesFinancierasBCRA.csv")

print("Estandarizando estructuras de IDs y Códigos con ceros a la izquierda...")
# 1.1 Blindar el BCRA: forzar strings limpios y aplicar zfill reglamentario
df_bcra = df_bcra.with_columns([
    pl.col("nro_id").cast(pl.Utf8).str.strip_chars().str.zfill(11),
    pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5)
])

# 1.2 Blindar el Padrón de ARCA: CUITs obligatorios de 11 caracteres
padron = padron.with_columns([
    pl.col("cuit").cast(pl.Utf8).str.strip_chars().str.zfill(11),
    pl.col("fecha_fallecimiento").str.strip_chars().alias("fallecimiento_clean"),
    pl.col("fecha_nacimiento").str.strip_chars().alias("nacimiento_clean"),
    pl.col("sexo").str.strip_chars().alias("sexo_clean")
])

# 1.3 Blindar el Maestro de Entidades: Códigos obligatorios de 5 caracteres
entidades_maestro = entidades_maestro.with_columns(
    pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5)
)

print("Ejecutando cruce poblacional (BCRA + Padrón)...")
# Usamos LEFT JOIN para el diagnóstico inicial de pérdidas
joined = df_bcra.join(padron, left_on="nro_id", right_on="cuit", how="left")

print("\n=======================================================")
print("📊 SUMMARY METRICS: PASO 1 (CRUCE Y POBLACIÓN PURIFICADA)")
print("=======================================================")

# Extraer métricas de control
metricas_join = joined.select([
    pl.len().alias("total_original"),
    pl.col("sexo_clean").is_not_null().sum().alias("conservados_join"),
    pl.col("sexo_clean").is_null().sum().alias("excluidos_no_padron"),
    (pl.col("nacimiento_clean") == "1901-01-01").sum().alias("fechas_inconsistentes"),
    ((pl.col("fallecimiento_clean") != "") & pl.col("fallecimiento_clean").is_not_null()).sum().alias("fallecidos_detectados"),
    (pl.col("sexo_clean") == "").sum().alias("sexo_vacio")
]).collect().to_pandas()

print(f"-> Total de registros originales BCRA: {metricas_join['total_original'][0]:,}")
print(f"-> Excluidos (Empresas / Sin Match de CUIT): {metricas_join['excluidos_no_padron'][0]:,}")
print(f"-> Conservados tras recuperar ceros iniciales: {metricas_join['conservados_join'][0]:,}")

print("\nAnomalías depuradas en los registros con match (Reglas ARCA):")
print(f"  - Fechas de nacimiento '1901-01-01' excluidas: {metricas_join['fechas_inconsistentes'][0]:,}")
print(f"  - Registros de personas fallecidas excluidos: {metricas_join['fallecidos_detectados'][0]:,}")
print(f"  - Género vacío excluido: {metricas_join['sexo_vacio'][0]:,}")

# Filtros sanitarios finales para el MCA
base_limpia = joined.filter(
    pl.col("sexo_clean").is_not_null()
).filter(
    (pl.col("fallecimiento_clean") == "") | pl.col("fallecimiento_clean").is_null()
).filter(
    pl.col("nacimiento_clean") != "1901-01-01"
).filter(
    pl.col("sexo_clean") != ""
)

total_final = base_limpia.select(pl.len()).collect().item()
print(f"\n=> 🏆 POBLACIÓN ACTIVA FINAL CONSERVADA POST-ZFILL: {total_final:,} deudores físicos")
print("=======================================================\n")

# Hacemos el Join definitivo con el maestro de nombres, ahora que ambos son de 5 dígitos estrictos
joined_con_nombres = base_limpia.join(entidades_maestro, on="cod_entidad", how="left")

1. Cargando bases de datos en modo Lazy...
Estandarizando estructuras de IDs y Códigos con ceros a la izquierda...
Ejecutando cruce poblacional (BCRA + Padrón)...

📊 SUMMARY METRICS: PASO 1 (CRUCE Y POBLACIÓN PURIFICADA)
-> Total de registros originales BCRA: 32,642,267
-> Excluidos (Empresas / Sin Match de CUIT): 28,335,300
-> Conservados tras recuperar ceros iniciales: 4,306,967

Anomalías depuradas en los registros con match (Reglas ARCA):
  - Fechas de nacimiento '1901-01-01' excluidas: 7,483
  - Registros de personas fallecidas excluidos: 11,990
  - Género vacío excluido: 39

=> 🏆 POBLACIÓN ACTIVA FINAL CONSERVADA POST-ZFILL: 4,287,465 deudores físicos



In [13]:
print("Escaneando la base cruda del BCRA...")
df_bcra = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/deudores_enero_2026_clasificados.parquet")

print("Calculando la distribución de longitudes de nro_id...")
# Limpiamos espacios y medimos cuántos caracteres tiene cada ID
profiling_id = df_bcra.with_columns(
    pl.col("nro_id").cast(pl.Utf8).str.strip_chars().str.len_chars().alias("longitud_id")
).group_by("longitud_id").len().sort("len", descending=True).collect().to_pandas()

# Calculamos el porcentaje para entender la gravedad
total_registros = profiling_id['len'].sum()
profiling_id['porcentaje (%)'] = (profiling_id['len'] / total_registros * 100).round(2)

print("\n--- RADIOGRAFÍA DE LOS IDs EN EL BCRA ---")
print(profiling_id.to_string(index=False))
print("-----------------------------------------")

Escaneando la base cruda del BCRA...
Calculando la distribución de longitudes de nro_id...

--- RADIOGRAFÍA DE LOS IDs EN EL BCRA ---
 longitud_id      len  porcentaje (%)
          11 32642267           100.0
-----------------------------------------


In [12]:
# ==========================================
print("2. Calculando concentración de deuda para el Top 20...")

# Identificar las 20 entidades que mayor volumen de deuda capturan en el sistema formal
top_20_calculado = joined_con_nombres.group_by(["cod_entidad", "nombre_entidad", "grupo_entidad"]).agg(
    pl.col("deuda_total").sum().alias("volumen_deuda_total")
).sort("volumen_deuda_total", descending=True).limit(20)

df_top20_ranking = top_20_calculado.collect().to_pandas()

# Filtrar la base maestra perezosa para conservar únicamente registros del Top 20
lista_codigos_top20 = df_top20_ranking["cod_entidad"].tolist()
base_filtrada_top20 = joined_con_nombres.filter(pl.col("cod_entidad").is_in(lista_codigos_top20))

# Realizar Feature Engineering sobre la población seleccionada
base_analitica_pl = base_filtrada_top20.with_columns(
    pl.col("nacimiento_clean").str.strptime(pl.Date, "%Y-%m-%d", strict=False).alias("fecha_nac_dt"),
    pl.col("codigo_postal").cast(pl.Int32, strict=False).fill_null(0).alias("cp_num")
).with_columns(
    ((pl.date(2026, 1, 1) - pl.col("fecha_nac_dt")).dt.total_days() / 365.25).floor().alias("edad")
).with_columns([
    pl.when(pl.col("edad") <= 25).then(pl.lit("Jóvenes (18-25)"))
    .when(pl.col("edad") <= 40).then(pl.lit("Adultos en Inserción (26-40)"))
    .when(pl.col("edad") <= 65).then(pl.lit("Adultos Consolidados (41-65)"))
    .otherwise(pl.lit("Adultos Mayores (>65)")).alias("Edad"),

    pl.when((pl.col("cp_num") >= 1000) & (pl.col("cp_num") <= 1499)).then(pl.lit("AMBA - CABA"))
    .when((pl.col("cp_num") >= 1600) & (pl.col("cp_num") <= 1999)).then(pl.lit("AMBA - Conurbano"))
    .otherwise(pl.lit("Interior y Resto del País")).alias("Geografia")
])

# Computar y materializar la base final
print("Materializando base consolidada...")
df_eda = base_analitica_pl.collect().to_pandas()

# --- IMPRESIÓN DE SUMMARY METRICS EXPLICATORIAS ---
print("\n=======================================================")
print("📊 SUMMARY METRICS: UNIVERSO TOP 20 ENTIDADES (ARCA-BCRA)")
print("=======================================================")
print(f"Total de registros deudores en el Top 20: {len(df_eda):,}")
print(f"Total de deuda administrada por el Top 20: ${df_eda['deuda_total'].sum():,.2f}")
print("\nDistribución de deudores por Grupo de Entidad:")
print(df_eda['grupo_entidad'].value_counts(normalize=True).round(4) * 100)
print("\nDistribución por Escala de Deuda:")
print(df_eda['segmento'].value_counts(normalize=True).round(4) * 100)
print("=======================================================\n")

2. Calculando concentración de deuda para el Top 20...
Materializando base consolidada...

📊 SUMMARY METRICS: UNIVERSO TOP 20 ENTIDADES (ARCA-BCRA)
Total de registros deudores en el Top 20: 3,333,460
Total de deuda administrada por el Top 20: $8,877,050,530.00

Distribución de deudores por Grupo de Entidad:
grupo_entidad
Banco                      59.26
Proveedor no Financiero    36.03
Compania Financiera         4.71
Name: proportion, dtype: float64

Distribución por Escala de Deuda:
segmento
Pequeño      98.87
Mediano       0.95
Grande        0.14
Sin Deuda     0.05
Name: proportion, dtype: float64



In [14]:
print(df_top20_ranking)

   cod_entidad                                     nombre_entidad  \
0        00011                       BANCO DE LA NACION ARGENTINA   
1        00007               BANCO DE GALICIA Y BUENOS AIRES S.A.   
2        00072                     BANCO SANTANDER ARGENTINA S.A.   
3        00014              BANCO DE LA PROVINCIA DE BUENOS AIRES   
4        00285                                   BANCO MACRO S.A.   
5        00017                          BANCO BBVA ARGENTINA S.A.   
6        70408                               TARJETA NARANJA S.A.   
7        00020              BANCO DE LA PROVINCIA DE CORDOBA S.A.   
8        00015  INDUSTRIAL AND COMMERCIAL BANK OF CHINA (ARGEN...   
9        72634                                MERCADOLIBRE S.R.L.   
10       00029                 BANCO DE LA CIUDAD DE BUENOS AIRES   
11       00027                             BANCO SUPERVIELLE S.A.   
12       00034                               BANCO PATAGONIA S.A.   
13       00330           NUEVO BAN

In [15]:
# ==========================================
# 📈 PASO 3: BATERÍA DE GRÁFICOS EXPLORATORIOS
# ==========================================
print("3. Generando visualizaciones descriptivas...")

# Gráfico 1: Concentración del Volumen de Deuda por Entidad Real
fig1 = px.bar(
    df_top20_ranking,
    x='volumen_deuda_total',
    y='nombre_entidad',
    color='grupo_entidad',
    orientation='h',
    title='Gráfico 1: Concentración de Deuda Total - Top 20 Entidades Financieras',
    labels={'volumen_deuda_total': 'Volumen de Deuda Acumulada ($)', 'nombre_entidad': 'Entidad Financiera', 'grupo_entidad': 'Grupo Institucional'},
    color_discrete_sequence=px.colors.qualitative.Dark24
)
fig1.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_white', height=600)
fig1.show()

# Gráfico 2: Composición del Tipo de Deuda según Grupo Institucional
df_g2 = df_eda.groupby(['grupo_entidad', 'segmento']).size().reset_index(name='Deudores')
fig2 = px.bar(
    df_g2, x='grupo_entidad', y='Deudores', color='segmento', barmode='stack',
    title='Gráfico 2: Composición de Escala de Deuda por Grupo de Entidad',
    labels={'grupo_entidad': 'Grupo Institucional', 'Deudores': 'Cantidad de Contratos', 'segmento': 'Escala de Deuda'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig2.update_layout(template='plotly_white')
fig2.show()

# Gráfico 3: Distribución de Edad según la Entidad del Top 20 (Heatmap Cruzado)
df_g3 = pd.crosstab(df_eda['nombre_entidad'], df_eda['Edad'], normalize='index') * 100
fig3 = px.imshow(
    df_g3, text_auto=".1f", color_continuous_scale='Purples',
    title='Gráfico 3: Perfil de Edad de las Carteras del Top 20 (% Horizontal por Banco)',
    labels=dict(x="Rango Etario del Deudor", y="Entidad Financiera", color="% de Cartera")
)
fig3.update_layout(template='plotly_white', height=600)
fig3.show()

# Gráfico 4: Asimetría Geográfica de las carteras por Grupo Institucional
df_g4 = pd.crosstab(df_eda['grupo_entidad'], df_eda['Geografia'], normalize='index') * 100
fig4 = px.bar(
    df_g4.reset_index().melt(id_vars='grupo_entidad'),
    x='value', y='grupo_entidad', color='Geografia', orientation='h',
    title='Gráfico 4: Segmentación Geográfica de las Carteras por Grupo Institucional (%)',
    labels={'value': 'Porcentaje de la Cartera (%)', 'grupo_entidad': 'Grupo Institucional', 'Geografia': 'Región'},
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig4.update_layout(template='plotly_white')
fig4.show()

3. Generando visualizaciones descriptivas...


In [26]:
# ==========================================
# 📊 PASO 3.5: DISTRIBUCIÓN MARGINAL DE CATEGORÍAS (MASAS)
# ==========================================
print("\n=======================================================")
print("🧮 DISTRIBUCIÓN DE CATEGORÍAS (INPUT PARA EL MCA)")
print("=======================================================")
print("Estas son las 'Masas' que alimentarán la inercia del modelo:\n")

# Definimos las columnas exactas que van al MCA
columnas_mca = ['nombre_entidad', 'segmento', 'sexo_clean', 'Edad', 'Geografia']
nombres_display = ['Entidad (Top 20)', 'Escala de Deuda', 'Género', 'Rango Etario', 'Región Geográfica']

# Calculamos y mostramos el porcentaje de cada categoría
for col, nombre in zip(columnas_mca, nombres_display):
    print(f"--- {nombre} ---")
    # Calculamos la frecuencia relativa (porcentaje)
    distribucion = df_eda[col].value_counts(normalize=True) * 100

    # Lo pasamos a un DataFrame para imprimirlo lindo
    df_dist = distribucion.reset_index()
    df_dist.columns = ['Categoría', 'Porcentaje (%)']
    df_dist['Porcentaje (%)'] = df_dist['Porcentaje (%)'].round(2).astype(str) + ' %'

    # Imprimimos sin el índice de Pandas
    print(df_dist.to_string(index=False))
    print("")

print("=======================================================\n")


🧮 DISTRIBUCIÓN DE CATEGORÍAS (INPUT PARA EL MCA)
Estas son las 'Masas' que alimentarán la inercia del modelo:

--- Entidad (Top 20) ---
                                                 Categoría Porcentaje (%)
                                       MERCADOLIBRE S.R.L.        21.41 %
                                      TARJETA NARANJA S.A.        14.62 %
                              BANCO DE LA NACION ARGENTINA         8.34 %
                      BANCO DE GALICIA Y BUENOS AIRES S.A.         7.73 %
                                 BANCO BBVA ARGENTINA S.A.         7.37 %
                                          BANCO MACRO S.A.         7.07 %
                     BANCO DE LA PROVINCIA DE BUENOS AIRES          6.8 %
                            BANCO SANTANDER ARGENTINA S.A.         6.78 %
                NARANJA DIGITAL COMPANIA FINANCIERA S.A.U.         4.71 %
INDUSTRIAL AND COMMERCIAL BANK OF CHINA (ARGENTINA) S.A.U.         2.26 %
                                      BANCO PATAG

In [28]:


# ... (sigue con la extracción de coordenadas y el gráfico fig5 igual que antes) ...

def clasificar_variable(attr):
    attr_str = str(attr)
    if attr_str.startswith('Entidad_'): return 'Top 20 Bancos/Fintech'
    # if attr_str.startswith('Escala_Deuda_'): return 'Escala Deuda' (Ya no lo necesitamos)
    if attr_str.startswith('Edad_'): return 'Rango Etario'
    if attr_str.startswith('Geografia_'): return 'Región Geográfica'
    if attr_str.startswith('Genero_'): return 'Género'
    return 'Otro'


4. Aplicando filtros de varianza y ejecutando MCA...


In [32]:
# ==========================================
# 🧠 PASO 4: PREPROCESAMIENTO Y MCA
# ==========================================
print("\n4. Aplicando filtros de varianza y ejecutando MCA...")

# 1. Eliminar outliers categóricos (Género X y categorías residuales)
df_mca_limpio = df_eda[df_eda['sexo_clean'].isin(['M', 'F'])].copy()

# 2. Seleccionar SOLO las variables con varianza real (Quitamos Escala_Deuda)
X_mca = df_mca_limpio[['nombre_entidad', 'sexo_clean', 'Edad', 'Geografia']].copy()

# Renombrar para que quede lindo en el gráfico
X_mca.columns = ['Entidad', 'Genero', 'Edad', 'Geografia']


# Entrenar el modelo
mca = prince.MCA(n_components=2, n_iter=10, random_state=42)
mca = mca.fit(X_mca)

v1, v2 = mca.percentage_of_variance_[0], mca.percentage_of_variance_[1]
print(f"¡MCA Finalizado! Varianza retenida: Dim1={v1:.2f}% | Dim2={v2:.2f}%")

# Extraer coordenadas limpias
coords = mca.column_coordinates(X_mca).copy()
coords.columns = ['Dim_1', 'Dim_2']
coords['Atributo_Original'] = coords.index

def limpiar_nombre(attr):
    prefijos = ['Entidad_', 'Escala_Deuda_', 'Edad_', 'Geografia_', 'Genero_']
    for p in prefijos:
        if str(attr).startswith(p): return str(attr).replace(p, '', 1)
    return str(attr)

def clasificar_variable(attr):
    attr_str = str(attr)
    if attr_str.startswith('Entidad_'): return 'Top 20 Bancos/Fintech'
    if attr_str.startswith('Escala_Deuda_'): return 'Escala Deuda'
    if attr_str.startswith('Edad_'): return 'Rango Etario'
    if attr_str.startswith('Geografia_'): return 'Región Geográfica'
    if attr_str.startswith('Genero_'): return 'Género'
    return 'Otro'

coords['Categoria'] = coords['Atributo_Original'].apply(limpiar_nombre)
coords['Tipo'] = coords['Atributo_Original'].apply(clasificar_variable)

# Gráfico 5: El Mapa Perceptual Definitivo de Líderes
fig5 = px.scatter(
    coords, x='Dim_1', y='Dim_2', color='Tipo', text='Categoria',
    title='Gráfico 5: Mapa Perceptual MCA - Top 20 Entidades y Atributos Demográficos',
    labels={'Dim_1': f"Dimensión 1 ({v1:.2f}%)", 'Dim_2': f"Dimensión 2 ({v2:.2f}%)"}
)
fig5.update_traces(textposition='top center', marker=dict(size=13, opacity=0.85, line=dict(width=1, color='DarkSlateGrey')))
fig5.update_layout(template='plotly_white', width=1300, height=850)
fig5.show()


4. Aplicando filtros de varianza y ejecutando MCA...
¡MCA Finalizado! Varianza retenida: Dim1=5.62% | Dim2=5.21%


In [17]:
# Guardar como HTML interactivo
archivo_html = "/content/drive/MyDrive/Tesis2026/Mapa_MCA_Top20.html"
fig5.write_html(archivo_html)
print(f"¡Gráfico interactivo guardado en: {archivo_html}!")

¡Gráfico interactivo guardado en: /content/drive/MyDrive/Tesis2026/Mapa_MCA_Top20.html!
